In [19]:
using LinearAlgebra
using Plots
using DataStructures

## Rotational and Radial Kinetic Energy

$$\alpha,\alpha^\prime \in \{1, \dots N\} \space | \space \beta,\beta^\prime \in [0, 2 \pi]$$

On Diagonal

$$T_{r \alpha \alpha^{\prime}} = \left(\frac{\hbar^2}{2\mu \Delta r^2 }\right) \cdot \left (-1\right)^{\left (\alpha - \alpha^{\prime} \right)} \cdot \left(\frac{\pi^2}{3} - \frac{1}{2\alpha^2}\right) $$

$${T}_{\phi \beta \beta^{\prime}} = \left(\frac{\hbar^2}{2\mu}\right) \cdot \left (-1\right)^{\left (\beta - \beta^{\prime} \right)} \cdot \left(\frac{N(N+1)}{3} \right)$$

On Diagonal 

$$T_{r \alpha \alpha^{\prime}} = \left(\frac{\hbar^2}{2\mu \Delta r^2}\right) \cdot \left(\frac{2}{(\alpha-\alpha^{\prime})^2} - \frac{2}{(\alpha+\alpha^{\prime})^2} \right)$$

$${T}_{\phi \beta \beta^{\prime}} = \left(\frac{\hbar^2}{2\mu}\right) \cdot \frac{cos\left(\frac{\pi(\beta-\beta)}{2N+1}\right)}{2sin\left(\frac{\pi(\beta-\beta)}{2N+1}\right)^2}$$



## DVR functions

In [20]:
function dvr_rotational(N, mu)
    h_bar = 1
    phi_grid = zeros((N))
    T = zeros((N, N))
    for a=1:N
        phi_grid[a] = (2*pi*(a-1)/(N))
        for ap=1:N
            if a==ap # On-diagonal
                T[a,ap] = h_bar^2 / (2*mu) * (-1)^(a-ap) * (N*(N+1)/3)
            else # Off-Diagonal
                T[a,ap] = h_bar^2 / (2*mu) * (-1)^(a-ap) * (cos(pi*(a-ap)/(2*N+1)))/(2*sin(pi*(a-ap)/(2*N+1))^2)
            
            end
        end
    end
    return T, phi_grid
end

function dvr_radial(N, r_min, r_max, mu)
    h_bar = 1
    r_grid = zeros((N))
    T = zeros((N, N))
    d_r = (r_max-r_min)/(N-1)
    for i=1:N
        r_grid[i] = d_r*(i-1) + r_min
    end
    for b=1:N
        for bp=1:N
            if b==bp # On-diagonal 
                T[b,bp] = h_bar^2 / (2*mu*d_r^2) * (-1)^(b-bp) * (pi^2/3 - 1/(2*b^2))
            else # Off-diagonal
                T[b,bp] = h_bar^2 / (2*mu*d_r^2) * (-1)^(b-bp) * (2/(b-bp)^2 - 2/(b+bp)^2)
            end
        end
    end
    return T, r_grid
end

dvr_radial (generic function with 1 method)

## Harmonic and Dipole-Dipole Potential

In $(r, \phi)$ basis

$$V_{\alpha} = \frac{1}{2} k \left(r_{\alpha} - r_e \right)^2$$

$$\hat{V}_{1,2} = \frac{\left(\frac{\mu}{r_e} \right)^2}{4 \pi \epsilon_o R^3} \cdot \hat{r}_1 \hat{r}_2 \cdot \left(sin (\hat{\phi}_1) sin (\hat{\phi}_2) - 2 cos (\hat{\phi}_1) cos (\hat{\phi}_2) \right)$$

In [21]:
function V_oscilation(N, k, r_e, r_grid)

    V = zeros((N))

    for i=1:N
        V[i] = 0.5*k * (r_grid[i] - r_e)^2
    end
    return V
end

function V_neighbours(N_r, N_phi, mu, r_grid, phi_grid, r_e, R_3)
    epsilon_o = 1
    size = N_r*N_phi
    V = zeros((size, size))
    i = 0
    C = ((mu/r_e)^2)/(4*pi*epsilon_o*R_3^3)
    C = 0
    for a1=1:N_r
        for a2=1:N_r
            i+=1
            j=0
            temp = C * r_grid[a1]*r_grid[a2]
            for b1=1:N_phi
                for b2=1:N_phi
                    j+=1
                    V[i,j] = temp * sin(phi_grid[b1])*sin(phi_grid[b2]) - 2*cos(phi_grid[b1])*cos(phi_grid[b2])
                end
            end
        end
    end
    return V
end

V_neighbours (generic function with 1 method)

## Constants for HF molecule

In [22]:
amu_to_au = 1/0.00054858 #au
ang_to_bohr = 1/0.529177 #1/bohr radius

wavenumber_to_Hartrees = 0.00000455633

m_H = 1.008 #in amu
m_F = 18.998 #in amu

r_e = 0.9168 * ang_to_bohr #in Angstrom

omega_e = 4138.385 #Hartrees

m_H *= amu_to_au
m_F *= amu_to_au

mu = (m_H*m_F)/(m_H+m_F) #reduced mass

omega_e *= wavenumber_to_Hartrees #h_bar = 1 (in au)

k = omega_e^2 * mu

Alpha = (mu*omega_e)/2 # h_bar not included as h_bar = 1

16.450694885570016

$$exp(-\frac{\mu \omega r^2}{2 \hbar})$$

For gaussian Distribution of vibrational states

$$\alpha (r-r_e)^2 = order$$

$$r = r_e \pm \sqrt{\frac{order}{\alpha}}

In [23]:
order = 5

r_min = r_e - sqrt(order/Alpha)
r_max = r_e + sqrt(order/Alpha)

2.2838078038699123

In [24]:
N_r = 24
N_phi = 24

mmax = 10

R = 10 #Distance between rotors

10

In [25]:
T_r, r_grid = dvr_radial(N_r, r_min, r_max, mu)

T_phi, phi_grid = dvr_rotational(N_phi, mu)

V_r = V_oscilation(N_r, k, r_e, r_grid)

24-element Vector{Float64}:
 0.09427923863524999
 0.07859573579989648
 0.06433800594957512
 0.05150604908428588
 0.04009986520402885
 0.03011945430880388
 0.021564816398611058
 0.014435951473450372
 0.008732859533321825
 0.004455540578225416
 0.0016039946081611459
 0.000178221623129014
 0.0001782216231290206
 0.0016039946081611656
 0.004455540578225433
 0.008732859533321848
 0.014435951473450401
 0.021564816398611093
 0.030119454308803965
 0.04009986520402885
 0.051506049084285944
 0.06433800594957519
 0.07859573579989655
 0.09427923863525009

## 1-body Hamiltonian, in the $(\alpha, m)$ basis

In [26]:
function Hamiltonian_1body(N_r, T_r, V_r, mu, m, r_grid)
    H = zeros((N_r, N_r))
    for a=1:N_r
        H[a,a] = V_r[a] + 1/(2*mu*r_grid[a]^2) * m^2
        for ap=1:N_r
            H[a,ap] += T_r[a,ap]
        end
    end
    return H
end

Hamiltonian_1body (generic function with 1 method)

$$\sum_{\alpha} \langle n_m | \alpha \rangle \cdot r_{\alpha} \langle \alpha | n'_{m'} \rangle$$

$$\langle e^{-im \phi} | \cos(\phi) | e^{im \phi} \rangle = \frac{1}{2} \left(\delta_{m, m' +1} + \delta_{m, m'-1} \right) = X_{n_m}$$

$$\langle e^{-im \phi} | \sin(\phi) | e^{im \phi} \rangle = \frac{1}{2} \left(\delta_{m, m' +1} - \delta_{m, m'-1} \right) = Y_{n_m}$$

A $2 \cdot (2 \space mmax + 1) \times 2 \cdot (2 \space mmax + 1)$ matrix will be constructed for each of the following

$$ \langle n_m | r \cos \phi | n_m' \rangle = \sum_{\alpha, m} \langle n | \alpha ; m \rangle r_{\alpha} \cdot \frac{1}{2} (\langle \alpha ; m+1 | n' \rangle + \langle \alpha ; m-1 | n' \rangle ) $$

$$ \langle n_m | r \sin \phi | n_m' \rangle = \sum_{\alpha, m} \langle n | \alpha ; m \rangle r_{\alpha} \cdot \frac{1}{2} (\langle \alpha ; m+1 | n' \rangle - \langle \alpha ; m-1 | n' \rangle ) $$

## Change of Basis: $(r, \phi) \to (\alpha, m)$

For each angular momentum quantum number $m$, the 1-body Hamiltonian $H^{(m)}$ is diagonalised in the radial DVR basis to give eigenstates $|n_m\rangle$, where $n$ indexes the vibrational levels.

The radial overlap between eigenstates in adjacent $m$ values is:

$$I^{(m \to m\pm 1)}_{n,n'} = \sum_{\alpha=1}^{N_r} \phi_{n,m}(\alpha) \cdot r_\alpha \cdot \phi_{n', m\pm 1}(\alpha)$$

Using $\cos\phi = \frac{1}{2}(e^{i\phi} + e^{-i\phi})$ and $\sin\phi = \frac{1}{2i}(e^{i\phi} - e^{-i\phi})$, and the angular momentum selection rule $\langle m | e^{\pm i\phi} | m' \rangle = \delta_{m', m \pm 1}$:

$$\langle n_m | r\cos\phi | n'_{m'} \rangle = \frac{1}{2}\left(\delta_{m', m+1} + \delta_{m', m-1}\right) I^{(m \to m')}_{n,n'}$$

$$\langle n_m | r\sin\phi | n'_{m'} \rangle = \frac{1}{2i}\left(\delta_{m', m+1} - \delta_{m', m-1}\right) I^{(m \to m')}_{n,n'}$$

Extracting real and imaginary parts yields the real matrices $X$ (cos, symmetric) and $Y$ (sin, antisymmetric), which are the local site operators will be passed to DMRG.

Basis ordering: states are indexed as $(n, m)$ with $n = 1, \dots, N_\text{states}$, over $m = -m_\text{max}, \dots, +m_\text{max}$. Total local Hilbert space dimension: $N_\text{states} \times (2\, m_\text{max} + 1)$.

3-fold recursion: at each $m$ step, eigenvectors for $m$, and $m+1$ are carried forward, computing $I^{(m \to m-1)}$ and $I^{(m+1 \to m)}$ without re-diagonalising at every iteration.

In [27]:
# Function version (can input range of energy states (n))

function basis_change(N_r, T_r, V_r, mu, mmax, N_min, N_max, r_grid)
    N_states = (N_max - N_min) + 1

    N_min += 1
    N_max += 1

    N_m = 2*mmax + 1 # -mmax to mmax
    N_mmax = N_states * N_m

    cos_matrix = zeros(Float64, N_mmax, N_mmax)
    sin_matrix = zeros(ComplexF64, N_mmax, N_mmax)
    eigenvalue_matrix = zeros(Float64, N_mmax, N_mmax)
    
    # Initialize 3-fold recursion ----------------------------------------
    h_m = Hamiltonian_1body(N_r, T_r, V_r, mu, -mmax, r_grid)
    evals_m = eigvals(Symmetric(h_m))[N_min:N_max]
    evec_m  = eigvecs(Symmetric(h_m))[:,N_min:N_max]

    h_m_1up = Hamiltonian_1body(N_r, T_r, V_r, mu, -mmax + 1, r_grid)
    evec_m_1up = eigvecs(Symmetric(h_m_1up))[:,N_min:N_max]

    evec_m_1down = 0
    # -------------------------------------------------------------------

    for m=-mmax:mmax
        m_idx = m + mmax # 0 to 2 mmax + 1

        # Loop over states |n> and |n'>
        for n=1:N_states
            row_idx = n + N_states * m_idx

            # Populate eigenvalue matrix
            eigenvalue_matrix[row_idx, row_idx] = evals_m[n]

            for np = 1:N_states
                delta_m_up, delta_m_down = 0 + 0im, 0 + 0im

                # Sum over all alpha
                for a=1:N_r
                    r = r_grid[a]
                    val_m = evec_m[a, n]

                    if m < mmax
                        delta_m_up += val_m * r * evec_m_1up[a, np]
                    end
                    if m > -mmax 
                        delta_m_down += val_m * r * evec_m_1down[a, np]
                    end
                end

                if m < mmax 
                    col_idx_up = np + N_states * (m_idx + 1)
                    cos_matrix[row_idx, col_idx_up] = 1/2 * delta_m_up
                    sin_matrix[row_idx, col_idx_up] = 1/2im * delta_m_up
                end

                if m > -mmax 
                    col_idx_down = np + N_states * (m_idx - 1)
                    cos_matrix[row_idx, col_idx_down] = 1/2 * delta_m_down
                    sin_matrix[row_idx, col_idx_down] = -1/2im * delta_m_down
                end
            end
        end

        # Shift 3-fold recursion variables for next loop iteration
        evec_m_1down = evec_m
        evec_m = evec_m_1up

        # Calculate next state
        if m < mmax - 1
            h_m_1up = Hamiltonian_1body(N_r, T_r, V_r, mu, m+2, r_grid)
            evec_m_1up = eigvecs(Symmetric(h_m_1up))[:,N_min:N_max]
            evals_m = eigvals(Symmetric(Hamiltonian_1body(N_r, T_r, V_r, mu, m + 1, r_grid)))[N_min:N_max]

        elseif m == mmax - 1
            evals_m = eigvals(Symmetric(Hamiltonian_1body(N_r, T_r, V_r, mu, mmax, r_grid)))[N_min:N_max]

        end
    end
    return cos_matrix, sin_matrix, eigenvalue_matrix
end

basis_change (generic function with 1 method)

In [28]:
# n=0,1
N_states = 2

N_m = 2*mmax + 1 # -mmax to mmax
N_max = N_states * N_m

cos_matrix = zeros(Float64, N_max, N_max)
sin_matrix = zeros(ComplexF64, N_max, N_max)
eigenvalue_matrix = zeros(Float64, N_max, N_max)

# Initialize 3-fold recursion ----------------------------------------
h_m = Hamiltonian_1body(N_r, T_r, V_r, mu, -mmax, r_grid)
evals_m = eigvals(Symmetric(h_m))[1:N_states]
evec_m  = eigvecs(Symmetric(h_m))[:,1:N_states]

h_m_1up = Hamiltonian_1body(N_r, T_r, V_r, mu, -mmax + 1, r_grid)
evec_m_1up = eigvecs(Symmetric(h_m_1up))[:,1:N_states]

evec_m_1down = 0
# -------------------------------------------------------------------

for m=-mmax:mmax
    m_idx = m + mmax # 0 to 2 mmax + 1

    # Loop over states |n> and |n'>
    for n=1:N_states
        row_idx = n + N_states * m_idx

        # Populate eigenvalue matrix
        eigenvalue_matrix[row_idx, row_idx] = evals_m[n]

        for np = 1:N_states
            delta_m_up, delta_m_down = 0 + 0im, 0 + 0im

            # Sum over all alpha
            for a=1:N_r
                r = r_grid[a]
                val_m = evec_m[a, n]

                if m < mmax
                    delta_m_up += val_m * r * evec_m_1up[a, np]
                end
                if m > -mmax 
                    delta_m_down += val_m * r * evec_m_1down[a, np]
                end
            end
            
            if m < mmax 
                col_idx_up = np + N_states * (m_idx + 1)
                cos_matrix[row_idx, col_idx_up] = 1/2 * delta_m_up
                sin_matrix[row_idx, col_idx_up] = 1/2im * delta_m_up
            end

            if m > -mmax 
                col_idx_down = np + N_states * (m_idx - 1)
                cos_matrix[row_idx, col_idx_down] = 1/2 * delta_m_down
                sin_matrix[row_idx, col_idx_down] = -1/2im * delta_m_down
            end
        end
    end

    # Shift 3-fold recursion variables for next loop iteration
    evec_m_1down = evec_m
    evec_m = evec_m_1up

    # Calculate next state
    if m < mmax - 1
        h_m_1up = Hamiltonian_1body(N_r, T_r, V_r, mu, m+2, r_grid)
        evec_m_1up = eigvecs(Symmetric(h_m_1up))[:,1:N_states]
        evals_m = eigvals(Symmetric(Hamiltonian_1body(N_r, T_r, V_r, mu, m + 1, r_grid)))[1:N_states]
        
    elseif m == mmax - 1
        evals_m = eigvals(Symmetric(Hamiltonian_1body(N_r, T_r, V_r, mu, mmax, r_grid)))[1:N_states]
        
    end
end

In [29]:
cos_matrix

42×42 Matrix{Float64}:
  0.0         0.0         0.874216   …   0.0         0.0         0.0
  0.0         0.0         0.0495034      0.0         0.0         0.0
  0.874216    0.0495034   0.0            0.0         0.0         0.0
 -0.0728844  -0.874351    0.0            0.0         0.0         0.0
  0.0         0.0        -0.872666       0.0         0.0         0.0
  0.0         0.0        -0.071817   …   0.0         0.0         0.0
  0.0         0.0         0.0            0.0         0.0         0.0
  0.0         0.0         0.0            0.0         0.0         0.0
  0.0         0.0         0.0            0.0         0.0         0.0
  0.0         0.0         0.0            0.0         0.0         0.0
  0.0         0.0         0.0        …   0.0         0.0         0.0
  0.0         0.0         0.0            0.0         0.0         0.0
  0.0         0.0         0.0            0.0         0.0         0.0
  ⋮                                  ⋱               ⋮          
  0.0         0

In [30]:
sin_matrix = imag.(sin_matrix)

sin_matrix

42×42 Matrix{Float64}:
  0.0         0.0        -0.874216   …   0.0         0.0         0.0
  0.0         0.0        -0.0495034      0.0         0.0         0.0
  0.874216    0.0495034   0.0            0.0         0.0         0.0
 -0.0728844  -0.874351    0.0            0.0         0.0         0.0
  0.0         0.0        -0.872666       0.0         0.0         0.0
  0.0         0.0        -0.071817   …   0.0         0.0         0.0
  0.0         0.0         0.0            0.0         0.0         0.0
  0.0         0.0         0.0            0.0         0.0         0.0
  0.0         0.0         0.0            0.0         0.0         0.0
  0.0         0.0         0.0            0.0         0.0         0.0
  0.0         0.0         0.0        …   0.0         0.0         0.0
  0.0         0.0         0.0            0.0         0.0         0.0
  0.0         0.0         0.0            0.0         0.0         0.0
  ⋮                                  ⋱               ⋮          
  0.0         0

In [ ]:
cos_matrix, sin_matrix, eigen_matrix = basis_change(N_r, T_r, V_r, mu, mmax, 0, 0, r_grid)

# Matrix for n=0, tri-diagonal as expected
cos_matrix

21×21 Matrix{Float64}:
 0.0        0.874216   0.0        0.0       …   0.0        0.0       0.0
 0.874216   0.0       -0.872666   0.0           0.0        0.0       0.0
 0.0       -0.872666   0.0       -0.871275      0.0        0.0       0.0
 0.0        0.0       -0.871275   0.0           0.0        0.0       0.0
 0.0        0.0        0.0        0.870046      0.0        0.0       0.0
 0.0        0.0        0.0        0.0       …   0.0        0.0       0.0
 0.0        0.0        0.0        0.0           0.0        0.0       0.0
 0.0        0.0        0.0        0.0           0.0        0.0       0.0
 0.0        0.0        0.0        0.0           0.0        0.0       0.0
 0.0        0.0        0.0        0.0           0.0        0.0       0.0
 0.0        0.0        0.0        0.0       …   0.0        0.0       0.0
 0.0        0.0        0.0        0.0           0.0        0.0       0.0
 0.0        0.0        0.0        0.0           0.0        0.0       0.0
 0.0        0.0        0.0  

In [38]:
cos_matrix, sin_matrix, eigen_matrix = basis_change(N_r, T_r, V_r, mu, mmax, 1, 1, r_grid)

# Matrix for n=1, tri-diagonal as expected
sin_matrix = imag.(sin_matrix)

21×21 Matrix{Float64}:
  0.0       0.874351   0.0        0.0       …   0.0        0.0       0.0
 -0.874351  0.0       -0.872772   0.0           0.0        0.0       0.0
  0.0       0.872772   0.0       -0.871353      0.0        0.0       0.0
  0.0       0.0        0.871353   0.0           0.0        0.0       0.0
  0.0       0.0        0.0        0.870099      0.0        0.0       0.0
  0.0       0.0        0.0        0.0       …   0.0        0.0       0.0
  0.0       0.0        0.0        0.0           0.0        0.0       0.0
  0.0       0.0        0.0        0.0           0.0        0.0       0.0
  0.0       0.0        0.0        0.0           0.0        0.0       0.0
  0.0       0.0        0.0        0.0           0.0        0.0       0.0
  0.0       0.0        0.0        0.0       …   0.0        0.0       0.0
  0.0       0.0        0.0        0.0           0.0        0.0       0.0
  0.0       0.0        0.0        0.0           0.0        0.0       0.0
  0.0       0.0        0.0  

In [ ]:
eigen_matrix #Diagonal matrix, n=1

21×21 Matrix{Float64}:
 0.0381807  0.0        0.0        …  0.0        0.0        0.0
 0.0        0.0363198  0.0           0.0        0.0        0.0
 0.0        0.0        0.0346481     0.0        0.0        0.0
 0.0        0.0        0.0           0.0        0.0        0.0
 0.0        0.0        0.0           0.0        0.0        0.0
 0.0        0.0        0.0        …  0.0        0.0        0.0
 0.0        0.0        0.0           0.0        0.0        0.0
 0.0        0.0        0.0           0.0        0.0        0.0
 0.0        0.0        0.0           0.0        0.0        0.0
 0.0        0.0        0.0           0.0        0.0        0.0
 0.0        0.0        0.0        …  0.0        0.0        0.0
 0.0        0.0        0.0           0.0        0.0        0.0
 0.0        0.0        0.0           0.0        0.0        0.0
 0.0        0.0        0.0           0.0        0.0        0.0
 0.0        0.0        0.0           0.0        0.0        0.0
 0.0        0.0        0.0      

In [32]:
# OLD CODE

"""mmax = 10

N_states = 2
N_m = 2*mmax + 1

N_max = N_states*N_m

cos_matrix = zeros(ComplexF64, N_max, N_max)
sin_matrix = zeros(ComplexF64, N_max, N_max)
eigenvalue_matrix = zeros(Float64, N_max, N_max)

h_m = Hamiltonian_1body(N_r, T_r, V_r, mu, -mmax)

evals_m = eigvals(h_m)[1:N_states]
evec_m = eigvecs(h_m)[:,1:N_states]

h_m_1up = Hamiltonian_1body(N_r, T_r, V_r, mu, -mmax+1)
evec_m_1up = eigvecs(h_m_1up)[:,1:N_states]

evec_m_1down = 0

for m=-mmax:mmax
    m_idx = m + mmax

    for n=1:N_states
        for np=1:N_states
            delta_m_up = 0 + 0im
            delta_m_down = 0 + 0im

            # Sum over all alpha values
            for a=1:N_r
                r = r_grid[a]
                val_m = evec_m[a,n]

                # Boundary edge cases
                # if m is not mmax
                if m < mmax
                    delta_m_up += val_m * r * evec_m_1up[a, np]
                end

                # If m is not -mmax
                if m > -mmax
                    delta_m_down += val_m * r * evec_m_1down[a, np]
                end
            end

            row_idx = n+N_states * m_idx  #alpha index

            # m index
            if m < mmax
                col_idx_up = np + N_states*(m_idx+1)
            end
            if m > -mmax
                col_idx_down = np + N_states*(m_idx-1)
            end
            
            if m == -mmax
                cos_matrix[row_idx, col_idx_up] = 1/2 * delta_m_up
                sin_matrix[row_idx, col_idx_up] = 1/2im * delta_m_up
            elseif m == mmax
                cos_matrix[row_idx, col_idx_down] = 1/2 * delta_m_down
                sin_matrix[row_idx, col_idx_down] = 1/2im * -delta_m_down
            else
                cos_matrix[row_idx, col_idx_up] = 1/2 * delta_m_up
                cos_matrix[row_idx, col_idx_down] = 1/2 * delta_m_down
                sin_matrix[row_idx, col_idx_up] = 1/2im * delta_m_up
                sin_matrix[row_idx, col_idx_down] = 1/2im * -delta_m_down
            end
        end
    end

    evec_m_1down = evec_m
    evec_m = evec_m_1up

    if m < mmax
        h_m_1up = Hamiltonian_1body(N_r, T_r, V_r, mu, m+1)
        evec_m_1up = eigvecs(h_m_1up)[:, 1:N_states]
    end
end

cos_matrix = real.(cos_matrix)
sin_matrix = imag.(sin_matrix)
"""

"mmax = 10\n\nN_states = 2\nN_m = 2*mmax + 1\n\nN_max = N_states*N_m\n\ncos_matrix = zeros(ComplexF64, N_max, N_max)\nsin_matrix = zeros(ComplexF64, N_max, N_max)\neigenvalue_matrix = zeros(Float64, N_max, N_max)\n\nh_m = Hamiltonian_1body(N_r, T_r, V_r, mu, -mmax)\n\nevals_m = eigva" ⋯ 1754 bytes ⋯ "  end\n    end\n\n    evec_m_1down = evec_m\n    evec_m = evec_m_1up\n\n    if m < mmax\n        h_m_1up = Hamiltonian_1body(N_r, T_r, V_r, mu, m+1)\n        evec_m_1up = eigvecs(h_m_1up)[:, 1:N_states]\n    end\nend\n\ncos_matrix = real.(cos_matrix)\nsin_matrix = imag.(sin_matrix)\n"

# Dipole-Dipole Interaction for DMRG (2-body)

$$D = \frac{1}{4 \pi \epsilon_o R^3} \frac{1}{N_r} \sum_{\alpha=1}^{N_r} \left(\frac{\mu}{r_\alpha}\right)^2$$

For the coplanar chain ($\gamma = 0^\circ$), the interaction between neighbouring rotors is:

$$\hat{V}_{i,i+1} = g \left[ Y_i Y_{i+1} - 2\, X_i X_{i+1} \right]$$

summed over $i = 1, \dots, N-1$ rotors in the chain, where $X =$ `cos_matrix` and $Y = $ `sin_matrix` are the local operators.

Order = 5, from gaussian distribution of vibrational states (seen above)

$$q = \frac{\mu_e}{r_e}, q_\alpha = \frac{\mu_e}{r_e} \cdot r_\alpha$$

$$\left(\frac{\mu}{r_e}\right)^2 = 0.172133$$

In [33]:
dipole_sum = 0

for i=1:N_r
    dipole_sum += (dipole_moment/r_grid[i])^2
end

dipole = (dipole_moment/r_e)^2

println(dipole_sum/N_r)
println(dipole)

0.19341569795064695
0.17213342977670262


In [34]:
dipole_moment = 0.718797 #au, from 1.827 Debye (D), NIST database
epsilon_0 = 1
R = 10 #au

dipole_sum = 0

for i=1:N_r
    dipole_sum += (dipole_moment/r_grid[i])^2
end

#dipole_per_r = (dipole_moment/r_e)^2 #

D = 1/(4 * pi * epsilon_0 * R^3) * 1/N_r * dipole_sum

1.5391532200207215e-5

In [35]:
# For a 2-rotor test: build the full 2-site interaction matrix via Kronecker product
# In DMRG, cos_matrix and sin_matrix are passed as local operators
# the tensor product Y_i ⊗ Y_{i+1} is handled automatically by the MPO framework.

V_12 = D * (kron(sin_matrix, sin_matrix) - 2 * kron(cos_matrix, cos_matrix))

println("2-site interaction matrix size: ", size(V_12))
println("Local operator (cos/sin matrix) size: ", size(cos_matrix))

2-site interaction matrix size: (1764, 1764)
Local operator (cos/sin matrix) size: (42, 42)
